# Convolutional Neural Networks with PyTorch

## What this notebook demonstrates

- CIFAR-10 image classification with convolutional neural networks
- bare PyTorch tensor implementations of convolutional models
- the nn.Module API for CNNs
- batch normalization and pooling layers
- training and evaluating CNN architectures

## Academic context

This notebook is a cleaned portfolio version of MSc coursework and implementation practice. It is included to demonstrate foundational computer vision and deep learning skills.


## GPU


If CUDA is available, the setup code uses it automatically; otherwise the notebook runs on CPU.


If CUDA is available, the setup code uses it automatically; otherwise the notebook runs on CPU.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import sampler

import torchvision.datasets as datasets
import torchvision.transforms as T

import numpy as np

USE_GPU = True
dtype = torch.float32 # We will be using float throughout this tutorial.

if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

# Constant to control how frequently we print train loss.
print_every = 100
print('using device:', device)


## Part I. Preparation


### Load CIFAR-10 dataset


Now, let's load the CIFAR-10 dataset.

This might take a couple minutes the first time you do it, but the files should stay cached after that.

In previous parts of the earlier notebooks we had to write our own code to download the CIFAR-10 dataset, preprocess it, and iterate through it in minibatches.

To make the whole procedure more efficient, we managed to save the whole dataset in ``.npy`` files.

PyTorch provides some convenient tools to automate this process even more for us.


In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Define the mean and standard deviation for CIFAR-10
mean = (0.4914, 0.4822, 0.4465)  # Precomputed mean for CIFAR-10
std = (0.2023, 0.1994, 0.2010)   # Precomputed std for CIFAR-10

# Define the transform for preprocessing and data augmentation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Load the CIFAR-10 dataset
dataset_path = "data"  
cifar10_dataset = datasets.CIFAR10(root=dataset_path, train=True, download=True, transform=transform) 

# Split the dataset into train, val, and test sets
num_train = 45000  # Number of training examples
num_val = 5000    # Number of validation examples
num_test = 10000   # Number of test examples (CIFAR-10 test set size)

train_dataset, val_dataset = random_split(cifar10_dataset, [num_train, num_val])

test_dataset = datasets.CIFAR10(root=dataset_path, train=False, download=True, transform=transform)

# Define DataLoaders
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Print dataset sizes for verification
print(f"Train set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")
print(f"Test set size: {len(test_dataset)}")


### Visualize the class distributions


We have now defined our train, validation and test dataloaders. However, it would be good to check the data distribution of each dataloader, in order to check how many examples per class are included. We do this as a sanity check, as we want our dataloaders to be distributed close to uniform. Let's define some helper functions:


In [ ]:
import numpy as np
from collections import Counter

def get_class_distribution(loader, dataset_classes):
    """
    Computes the distribution of classes in a DataLoader.

    Args:
    - loader: DataLoader to inspect
    - dataset_classes: List of class names for the dataset

    Returns:
    - class_counts: Dictionary with class names as keys and counts as values
    """
    class_counts = Counter()
    for inputs, labels in loader:
        class_counts.update(labels.numpy())

    # Convert to a dictionary with class names
    class_distribution = {dataset_classes[i]: class_counts[i] for i in range(len(dataset_classes))}
    return class_distribution

def display_class_distribution(distribution, title="Class Distribution"):
    """
    Displays the class distribution as a bar chart.

    Args:
    - distribution: Dictionary with class names as keys and counts as values
    - title: Title for the bar chart
    """
    import matplotlib.pyplot as plt

    class_names = list(distribution.keys())
    counts = list(distribution.values())

    plt.figure(figsize=(10, 6))
    plt.bar(class_names, counts)
    plt.title(title)
    plt.xlabel("Class")
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.show()


Now, let's visualize the distribution of the examples per class for each dataloader:


In [ ]:
# CIFAR-10 classes
cifar10_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

# Compute and display class distribution for each DataLoader
for loader, name in [(train_loader, "Train"), (val_loader, "Validation"), (test_loader, "Test")]:
    distribution = get_class_distribution(loader, cifar10_classes)
    print(f"{name} DataLoader Distribution:")
    for cls, count in distribution.items():
        print(f"  {cls}: {count}")
    display_class_distribution(distribution, title=f"{name} DataLoader Class Distribution")


### Visualize CIFAR-10 images


Now let's visualize a few examples from each class in the CIFAR-10 dataset.

By displaying images alongside their corresponding class labels, we can verify the dataset’s contents and ensure the classes are represented correctly.

This visualization helps us understand the data distribution and provides insights into the dataset’s characteristics.


In [ ]:
import matplotlib.pyplot as plt

def visualize_images_per_class(dataset, dataset_classes, num_images=5):
    """
    Visualizes a few images from each class in the dataset.

    Args:
    - dataset: The dataset to sample from.
    - dataset_classes: List of class names for the dataset.
    - num_images: Number of images to display per class.
    """
    class_to_indices = {cls: [] for cls in range(len(dataset_classes))}

    # Group indices by class
    for idx, (_, label) in enumerate(dataset):
        if len(class_to_indices[label]) < num_images:
            class_to_indices[label].append(idx)
        if all(len(indices) == num_images for indices in class_to_indices.values()):
            break

    # Plot the images
    fig, axes = plt.subplots(len(dataset_classes), num_images + 1, figsize=((num_images + 1) * 2, len(dataset_classes) * 2))
    for cls_idx, cls_name in enumerate(dataset_classes):
        indices = class_to_indices[cls_idx]

        # Add class name to the first column of each row
        axes[cls_idx, 0].text(0.5, 0.5, cls_name, fontsize=12, ha='center', va='center', transform=axes[cls_idx, 0].transAxes)
        axes[cls_idx, 0].axis("off")

        for img_idx, data_idx in enumerate(indices):
            img, label = dataset[data_idx]
            ax = axes[cls_idx, img_idx + 1]  # Shift by 1 for the class name column
            img = img.permute(1, 2, 0)  # Convert CHW to HWC
            img = img * torch.tensor(std).view(1, 1, 3) + torch.tensor(mean).view(1, 1, 3)  # De-normalize
            img = torch.clamp(img, 0, 1)  # Clip values to valid range
            ax.imshow(img)
            ax.axis("off")
    plt.tight_layout()
    plt.show()

# Visualize 5 images per class for CIFAR-10 train set
visualize_images_per_class(cifar10_dataset, cifar10_classes, num_images=5)


## Part II. Three-Layer ConvNet with Barebone PyTorch


### Barebone PyTorch: Three-Layer ConvNet


**Implementation note.**

This section defines the function `three_layer_convnet`, which performs the forward pass of a three-layer convolutional network. The implementation is checked our implementation by passing zeros through the network. The network has the following architecture:

1. A convolutional layer (with bias) with `channel_1` filters, each with shape `KW1 x KH1`, and zero-padding of two
2. ReLU nonlinearity
3. A convolutional layer (with bias) with `channel_2` filters, each with shape `KW2 x KH2`, and zero-padding of one
4. ReLU nonlinearity
5. Fully-connected layer with bias, producing scores for C classes.

Note that we have **no softmax activation** here after our fully-connected layer: this is because PyTorch's cross entropy loss performs a softmax activation for you, and by bundling that step in makes computation more efficient.

**HINT**: For convolutions: http://pytorch.org/docs/stable/nn.html#torch.nn.functional.conv2d; pay attention to the shapes of convolutional filters!


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

def three_layer_convnet(x, params):
    """
    Performs the forwar d pass of a three-layer convolutional network with the
    architecture defined above.

    Inputs:
    - x: A PyTorch Tensor of shape (N, 3, H, W) giving a minibatch of images
    - params: A list of PyTorch Tensors giving the weights and biases for the
      network; should contain the following:
      - conv_w1: PyTorch Tensor of shape (channel_1, 3, KH1, KW1) giving weights
        for the first convolutional layer
      - conv_b1: PyTorch Tensor of shape (channel_1,) giving biases for the first
        convolutional layer
      - conv_w2: PyTorch Tensor of shape (channel_2, channel_1, KH2, KW2) giving
        weights for the second convolutional layer
      - conv_b2: PyTorch Tensor of shape (channel_2,) giving biases for the second
        convolutional layer
      - fc_w: PyTorch Tensor giving weights for the fully-connected layer. Can you
        figure out what the shape should be?
      - fc_b: PyTorch Tensor giving biases for the fully-connected layer. Can you
        figure out what the shape should be?

    Returns:
    - scores: PyTorch Tensor of shape (N, C) giving classification scores for x
    """
    conv_w1, conv_b1, conv_w2, conv_b2, fc_w, fc_b = params

    scores = None
    x = F.conv2d(x, weight=conv_w1, bias=conv_b1, padding=2)
    x = F.relu(x)
    x = F.conv2d(x, weight=conv_w2, bias=conv_b2, padding=1)
    x = F.relu(x)
    x = torch.flatten(x, start_dim=1)
    x = x @ fc_w + fc_b

    scores = x
    return scores


After defining the forward pass of the ConvNet above, run the following cell to validate the implementation.

When you run this function, scores should have shape (64, 10).


In [ ]:
def three_layer_convnet_test():
    x = torch.zeros((64, 3, 32, 32), dtype=dtype)  # minibatch size 64, image size [3, 32, 32]

    conv_w1 = torch.zeros((6, 3, 5, 5), dtype=dtype)  # [out_channel, in_channel, kernel_H, kernel_W]
    conv_b1 = torch.zeros((6,))  # out_channel
    conv_w2 = torch.zeros((9, 6, 3, 3), dtype=dtype)  # [out_channel, in_channel, kernel_H, kernel_W]
    conv_b2 = torch.zeros((9,))  # out_channel

    # you must calculate the shape of the tensor after two conv layers, before the fully-connected layer
    fc_w = torch.zeros((9 * 32 * 32, 10))
    fc_b = torch.zeros(10)

    scores = three_layer_convnet(x, [conv_w1, conv_b1, conv_w2, conv_b2, fc_w, fc_b])
    print(scores.size())  # expected shape [64, 10]
three_layer_convnet_test()


### Barebone PyTorch: Initialization


Let's write a couple utility methods to initialize the weight matrices for our models.

- `random_weight(shape)` initializes a weight tensor with the Kaiming normalization method.
- `zero_weight(shape)` initializes a weight tensor with all zeros. Useful for instantiating bias parameters.

The `random_weight` function uses the Kaiming normal initialization method, described in:

He et al, *Delving Deep into Rectifiers: Surpassing Human-Level Performance on ImageNet Classification*, ICCV 2015, https://arxiv.org/abs/1502.01852


In [ ]:
def random_weight(shape):
    """
    Create random Tensors for weights; setting requires_grad=True means that we
    want to compute gradients for these Tensors during the backward pass.
    We use Kaiming normalization: sqrt(2 / fan_in)
    """
    if len(shape) == 2:  # FC weight
        fan_in = shape[0]
    else:
        fan_in = np.prod(shape[1:]) # conv weight [out_channel, in_channel, kH, kW]
    # randn is standard normal distribution generator.
    w = torch.randn(shape, device=device, dtype=dtype) * np.sqrt(2. / fan_in)
    w.requires_grad = True
    return w

def zero_weight(shape):
    return torch.zeros(shape, device=device, dtype=dtype, requires_grad=True)

# create a weight of shape [3 x 5]
# expected shape the type `torch.cuda.FloatTensor` if you use GPU.
# Otherwise it should be `torch.FloatTensor`
random_weight((3, 5))


### Barebone PyTorch: Check Accuracy


When training the model we will use the following function to check the accuracy of our model on the training or validation sets.

When checking accuracy we don't need to compute any gradients; as a result we don't need PyTorch to build a computational graph for us when we compute scores. To prevent a graph from being built we scope our computation under a `torch.no_grad()` context manager.


In [ ]:
def check_accuracy_part2(loader, model_fn, params):
    """
    Check the accuracy of a classification model.

    Inputs:
    - loader: A DataLoader for the data split we want to check
    - model_fn: A function that performs the forward pass of the model,
      with the signature scores = model_fn(x, params)
    - params: List of PyTorch Tensors giving parameters of the model

    Returns: Nothing, but prints the accuracy of the model
    """
    #split = 'val' if loader.dataset.train else 'test'
    split = 'val' if len(loader) == 79 else 'test'
    print('Checking accuracy on the %s set' % split)
    num_correct, num_samples = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)  # move to device, e.g. GPU
            y = y.to(device=device, dtype=torch.int64)
            scores = model_fn(x, params)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
        acc = float(num_correct) / num_samples
        print('Got %d / %d correct (%.2f%%)' % (num_correct, num_samples, 100 * acc))


### BareBone PyTorch: Training Loop


We can now set up a basic training loop to train our network. We will train the model using stochastic gradient descent without momentum. We will use `torch.functional.cross_entropy` to compute the loss; you can [read about it here](http://pytorch.org/docs/stable/nn.html#cross-entropy).

The training loop takes as input the neural network function, a list of initialized parameters (`[w1, w2]` in our example), and learning rate.


In [ ]:
def train_part2(model_fn, params, learning_rate):
    """
    Train a model on CIFAR-10.

    Inputs:
    - model_fn: A Python function that performs the forward pass of the model.
      It should have the signature scores = model_fn(x, params) where x is a
      PyTorch Tensor of image data, params is a list of PyTorch Tensors giving
      model weights, and scores is a PyTorch Tensor of shape (N, C) giving
      scores for the elements in x.
    - params: List of PyTorch Tensors giving weights for the model
    - learning_rate: Python scalar giving the learning rate to use for SGD

    Returns: Nothing
    """
    for t, (x, y) in enumerate(train_loader):
        # Move the data to the proper device (GPU or CPU)
        x = x.to(device=device, dtype=dtype)
        y = y.to(device=device, dtype=torch.long)

        # Forward pass: compute scores and loss
        scores = model_fn(x, params)
        loss = F.cross_entropy(scores, y)

        # Backward pass: PyTorch figures out which Tensors in the computational
        # graph has requires_grad=True and uses backpropagation to compute the
        # gradient of the loss with respect to these Tensors, and stores the
        # gradients in the .grad attribute of each Tensor.
        loss.backward()

        # Update parameters. We don't want to backpropagate through the
        # parameter updates, so we scope the updates under a torch.no_grad()
        # context manager to prevent a computational graph from being built.
        with torch.no_grad():
            for w in params:
                w -= learning_rate * w.grad

                # Manually zero the gradients after running the backward pass
                w.grad.zero_()

        if t % print_every == 0:
            print('Iteration %d, loss = %.4f' % (t, loss.item()))
            check_accuracy_part2(val_loader, model_fn, params)
            print()


### BareBone PyTorch: Training a ConvNet


**Implementation note.**

Use the functions defined above to train a three-layer convolutional network on CIFAR-10. The network has the following architecture:

1. Convolutional layer (with bias) with 32 5x5 filters, with zero-padding of 2
2. ReLU
3. Convolutional layer (with bias) with 16 3x3 filters, with zero-padding of 1
4. ReLU
5. Fully-connected layer (with bias) to compute scores for 10 classes

Initialize your weight matrices using the `random_weight` function defined above, and initialize your bias vectors using the `zero_weight` function above.

No hyperparameter tuning is required for this baseline.

Train for one epoch..


In [ ]:
learning_rate = 3e-4

channel_1 = 32
channel_2 = 16

conv_w1 = random_weight((channel_1, 3, 5, 5))  # [out_channel, in_channel, kernel_H, kernel_W]
conv_b1 = zero_weight((channel_1,))
conv_w2 = random_weight((channel_2, 32, 3, 3))
conv_b2 = zero_weight((channel_2,))
fc_w = random_weight((channel_2 * 32 * 32, 10))
fc_b = zero_weight((10,))

params = [conv_w1, conv_b1, conv_w2, conv_b2, fc_w, fc_b]
train_part2(three_layer_convnet, params, learning_rate)


## Part III. Three-Layer ConvNet with PyTorch Module API


### Module API: Three-Layer ConvNet


**Implementation note.**

Now, let's implement a 3-layer ConvNet followed by a fully connected layer using the `Module API`. The network architecture should be the same as in Part II:

1. Convolutional layer with `channel_1` 5x5 filters with zero-padding of 2
2. ReLU
3. Convolutional layer with `channel_2` 3x3 filters with zero-padding of 1
4. ReLU
5. Fully-connected layer to `num_classes` classes

Initialize the weight matrices of the model using the Kaiming normal initialization method.

**HINT**: http://pytorch.org/docs/stable/nn.html#conv2d

After defining the three-layer ConvNet, the `test_ThreeLayerConvNet` function will run the implementation; it should print `(64, 10)` for the shape of the output scores.


In [ ]:
class ThreeLayerConvNet(nn.Module):
    def __init__(self, in_channel, channel_1, channel_2, num_classes):
        super().__init__()
        # architecture defined above.                                          #
        self.conv1 = nn.Conv2d(in_channels=in_channel, out_channels=channel_1, kernel_size=5, padding=2)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv2d(in_channels=channel_1, out_channels=channel_2, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.fc = nn.Linear(in_features=channel_2 * 32 * 32, out_features=num_classes)


        self.conv1.weight = nn.Parameter(random_weight(self.conv1.weight.shape))
        self.conv2.weight = nn.Parameter(random_weight(self.conv2.weight.shape))
        self.fc.weight = nn.Parameter(random_weight(self.fc.weight.shape))

        nn.init.zeros_(self.conv1.bias)
        nn.init.zeros_(self.conv2.bias)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x):
        scores = None
        # should use the layers you defined in __init__ and specify the        #
        # connectivity of those layers in forward()                            #
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.conv2(x)
        x = self.relu2(x)
        x = torch.flatten(x, start_dim=1)
        x = self.fc(x)

        scores = x
        return scores


def test_ThreeLayerConvNet():
    x = torch.zeros((64, 3, 32, 32), device=device, dtype=dtype)  # minibatch size 64, image size [3, 32, 32]
    model = ThreeLayerConvNet(in_channel=3, channel_1=12, channel_2=8, num_classes=10).to(device)
    scores = model(x)
    print(scores.size())  # expected shape [64, 10]
test_ThreeLayerConvNet()


### Module API: Check Accuracy


Given the validation or test set, we can check the classification accuracy of a neural network.

This version is slightly different from the one in part II. You don't manually pass in the parameters anymore.


In [ ]:
def check_accuracy_part3(loader, model):
    split = 'val' if len(loader) == 79 else 'test'
    print('Checking accuracy on the %s set' % split)
    num_correct, num_samples = 0, 0
    model.eval()  # set model to evaluation mode
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)  # move to device, e.g. GPU
            y = y.to(device=device, dtype=torch.long)
            scores = model(x)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
        acc = float(num_correct) / num_samples
        print('Got %d / %d correct (%.2f)' % (num_correct, num_samples, 100 * acc))


### Module API: Training Loop


We also use a slightly different training loop. Rather than updating the values
of the weights ourselves, we use an Optimizer object from the `torch.optim`
package, which abstract the notion of an optimization algorithm and provides implementations of most of the algorithms commonly used to optimize neural networks.


In [ ]:
def train_part3(model, optimizer, epochs=1):
    """
    Train a model on EuroSAT using the PyTorch Module API.

    Inputs:
    - model: A PyTorch Module giving the model to train.
    - optimizer: An Optimizer object we will use to train the model
    - epochs: (Optional) A Python integer giving the number of epochs to train for

    Returns: Nothing, but prints model accuracies during training.
    """
    epoch_loss = 0

    model = model.to(device=device)  # move the model parameters to CPU/GPU
    for e in range(epochs):
        for t, (x, y) in enumerate(train_loader):
            model.train()  # put model to training mode
            x = x.to(device=device, dtype=dtype)  # move to device, e.g. GPU
            y = y.to(device=device, dtype=torch.long)

            scores = model(x)
            loss = F.cross_entropy(scores, y)

            epoch_loss += loss.item()

            # Zero out all of the gradients for the variables which the optimizer
            # will update.
            optimizer.zero_grad()

            # This is the backwards pass: compute the gradient of the loss with
            # respect to each  parameter of the model.
            loss.backward()

            # Actually update the parameters of the model using the gradients
            # computed by the backwards pass.
            optimizer.step()

            if t % print_every == 0:
                print('Iteration %d, loss = %.4f' % (t, loss.item()))
                check_accuracy_part3(val_loader, model)
                print()

        epoch_loss /= train_loader.batch_size
    return epoch_loss


### Module API: Train a Three-Layer ConvNet


**Implementation note.**

Use the Module API to train a three-layer ConvNet on CIFAR-10. This follows the same structure as the two-layer-network training loop. No hyperparameter tuning is required for this baseline.

Train for one epoch..

Train the model using stochastic gradient descent without momentum.


In [ ]:
learning_rate = 3e-4
channel_1 = 32
channel_2 = 16

model = None
optimizer = None
model = ThreeLayerConvNet(3, channel_1, channel_2, 10)
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

_ = train_part3(model, optimizer)


## Part IV. Three-Layer ConvNet with PyTorch Sequential API


### Sequential API: Three-Layer ConvNet


**Implementation note.**

Use `nn.Sequential` to define and train a three-layer ConvNet with the same architecture we used in Part III:

1. Convolutional layer (with bias) with 32 5x5 filters, with zero-padding of 2
2. ReLU
3. Convolutional layer (with bias) with 16 3x3 filters, with zero-padding of 1
4. ReLU
5. Fully-connected layer (with bias) to compute scores for 10 classes

The default PyTorch weight initialization is used.

Optimize your model using stochastic gradient descent with Nesterov momentum 0.9.

Again, you don't need to tune any hyperparameters.

Train for one epoch..


In [ ]:
channel_1 = 32
channel_2 = 16
learning_rate = 3e-4
num_classes = 10
in_channel = 3

model = None
optimizer = None


# Sequential API.                                                              #

model = nn.Sequential(
    nn.Conv2d(in_channels=in_channel, out_channels=channel_1, kernel_size=5, padding=2),
    nn.ReLU(),
    nn.Conv2d(in_channels=channel_1, out_channels=channel_2, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Flatten(start_dim=1, end_dim=-1),
    nn.Linear(in_features=channel_2 * 32 * 32, out_features=num_classes)
)
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, nesterov=True)

train_part3(model, optimizer)


### Sequential API: Experiment with Optimizers


**Implementation note.**

Given the baseline three-layer convolutional network model implemented with PyTorch's Sequential API (as defined above), this section train this model for 1 epoch using as **optimizers**: `SGD with momentum`, `Adagrad`, `RMSProp`, `Adam`, `AdamW` and `NAdam`, similar to what you did in the previous notebook.

1. Refer to the PyTorch documentation about [optimizers](https://pytorch.org/docs/stable/optim.html.) to implement these optimizers.

2. Perform a learning rate search for each optimizer to find the learning rate that yields the best accuracy on the validation set.

For faster iteration:

- Use `channel_1 = 32`, `channel_2 = 16`.

- Utilize the `train_part3` function for training.

NOTE: You'll probably have to modify the `check_accuracy_part3()` function to return the computed accuracy (as a percentage).


In [ ]:
def check_accuracy_part3(loader, model):
    """
    Check the accuracy of a classification model using nn.Module API.

    Inputs:
    - loader: A DataLoader for the data split we want to check.
    - model: A PyTorch model instance (subclass of nn.Module).

    Returns:
    - acc: Accuracy of the model on the dataset (as a percentage).
    """
    split = 'val' if len(loader) == 79 else 'test'
    print('Checking accuracy on the %s set' % split)

    num_correct, num_samples = 0, 0
    model.eval()  # Set the model to evaluation mode
    with torch.no_grad():  # Disable gradient computation
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)  # Move input to device
            y = y.to(device=device, dtype=torch.long)  # Move labels to device with correct dtype

            # Forward pass using the model
            scores = model(x)
            _, preds = scores.max(1)  # Get class predictions

            num_correct += (preds == y).sum().item()  # Increment correct predictions
            num_samples += preds.size(0)  # Increment total samples

    acc = (float(num_correct) / num_samples) * 100
    print('Got %d / %d correct (%.2f%%)' % (num_correct, num_samples, acc))
    return acc


In [ ]:
in_channel = 3
num_classes = 10
channel_1 = 32
channel_2 = 16
learning_rates = [1e-1, 1e-2, 1e-3, 1e-4]

# Define the optimizers to test
optimizers_to_test = {
    "SGD": lambda model, lr: torch.optim.SGD(model.parameters(), lr=lr),
    "SGD with momentum": lambda model, lr: torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True),
    "Adagrad": lambda model, lr: torch.optim.Adagrad(model.parameters(), lr=lr),
    "RMSProp": lambda model, lr: torch.optim.RMSprop(model.parameters(), lr=lr),
    "Adam": lambda model, lr: torch.optim.Adam(model.parameters(), lr=lr),
    "AdamW": lambda model, lr: torch.optim.AdamW(model.parameters(), lr=lr),
    "NAdam": lambda model, lr: torch.optim.NAdam(model.parameters(), lr=lr),
}

torch.manual_seed(42)

results = {}

# rates. Use `train_part3()` for training and `check_accuracy_part3()` to      #
# evaluate validation accuracy.                                                #
for optimizer_name, optimizer_fn in optimizers_to_test.items():

    best_loss = float('inf')
    best_lr = None
    for lr in learning_rates:
        model = nn.Sequential(
            nn.Conv2d(in_channels=in_channel, out_channels=channel_1, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv2d(in_channels=channel_1, out_channels=channel_2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten(start_dim=1, end_dim=-1),
            nn.Linear(in_features=channel_2 * 32 * 32, out_features=num_classes)
        )
        model.to(device=device)
        optimizer = optimizer_fn(model, lr)

        print(f"Testing optimizer: {optimizer_name} with learning rate: {lr}")
        epoch_loss = train_part3(model, optimizer)

        acc = check_accuracy_part3(val_loader, model)

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_lr = lr

    results[optimizer_name] = (best_lr, acc)

for optimizer_name, eval in results.items():
    best_lr = eval[0]
    acc = eval[1]
    print(f'Optimizer: {optimizer_name}, Best Learning Rate: {best_lr}, Validation Accuracy: {acc}')


Keep the best classification results (based on validation accuracy) per optimizer in the following Table:


| Optimizer | best LR | Validation Accuracy |
|----------|----------|----------|
| SGD    | 0.01   | 21.4   |
| SGD + momentum    | 0.01   | 34.32   |
| Adagrad    | 0.01   | 35.5   |
| RMSProp    | 0.0001   | 48.28   |
| Adam    | 0.001   | 52.44   |
| AdamW    | 0.001   | 50.76   |
| NAdam    | 0.001   | 48.26   |


### Sequential API: Experiment with Activation Functions


**Implementation note.**

Using the optimizer and learning rate that performed the best in the previous question, train the three-layer convolutional network for 1 epoch using different **activation functions**: `Leaky ReLU`, `ELU`, `GeLU`, `PReLU`, `SiLU` and `Mish`, similar to what you did in the previous notebook.

1. Refer to the PyTorch documentation about [non-linear activations](https://pytorch.org/docs/stable/nn.html#non-linear-activations-weighted-sum-nonlinearity) to implement these activation functions.

2. Observe the validation accuracy for each activation function and select the one that performs best.

For faster iteration:

- Use `channel_1 = 32`, `channel_2 = 16`.

- Utilize the `train_part3` function for training.


In [ ]:
channel_1 = 32
channel_2 = 16
best_learning_rate = 0.001  # Replace with the best learning rate from previous experiments
best_optimizer_fn = lambda model: torch.optim.Adam(model.parameters(), lr=best_learning_rate)  # Replace with best optimizer

# Define activation functions to test
activation_functions = {
    "ReLU": nn.ReLU(),
    "Leaky ReLU": nn.LeakyReLU(negative_slope=0.01),
    "ELU": nn.ELU(),
    "GeLU": nn.GELU(),
    "PReLU": nn.PReLU(),
    "SiLU": nn.SiLU(),
    "Mish": nn.Mish(),
}

results = {}

# `train_part3()` for training and `check_accuracy_part3()` to evaluate        #
# validation accuracy.                                                         #
for activation_name, activation_fn in activation_functions.items():

    model = nn.Sequential(
        nn.Conv2d(in_channels=in_channel, out_channels=channel_1, kernel_size=5, padding=2),
        activation_fn,
        nn.Conv2d(in_channels=channel_1, out_channels=channel_2, kernel_size=3, padding=1),
        activation_fn,
        nn.Flatten(start_dim=1, end_dim=-1),
        nn.Linear(in_features=channel_2 * 32 * 32, out_features=num_classes)
    )
    model.to(device=device)
    optimizer = best_optimizer_fn(model)

    print(f"Testing activation function: {activation_name}.")
    epoch_loss = train_part3(model, optimizer)

    acc = check_accuracy_part3(val_loader, model)

    results[activation_name] = acc

for activation_name, acc in results.items():
    print(f'Activation: {activation_name}, Validation Accuracy: {acc}')


Keep the classification results in the following Table:


| Activation function | Validation Accuracy |
|----------|----------|
| ReLU    | 59.09   |
| Leaky ReLU    | 57.87   |
| ELU    | 48.18   |
| GeLU    | 56.18   |
| PReLU    | 56.89   |
| SiLU    | 53.18   |
| Mish    | 53.26   |


### Sequential API: Experiment with Channels Sizes (number of filters)


**Implementation note.**

Using the best optimizer, learning rate, and activation function from the previous experiments, train the three-layer convolutional network for 1 epoch with different channel sizes. Experiment with the following channel sizes for the convolutional layers:

- `channel_1 = 16`, `channel_2 = 8`
- `channel_1 = 32`, `channel_2 = 16`
- `channel_1 = 64`, `channel_2 = 32`

1.	Train the model for 1 epoch for each configuration and record the train and validation accuracy.

2.	Plot a graph with the channel sizes (e.g., channel_1) on the x-axis and the train/validation accuracy on the y-axis.


In [ ]:
# Define the channel size configurations to test
channel_sizes = [
    (16, 8),
    (32, 16),
    (64, 32)
]

best_learning_rate = 0.001  # Replace with the best learning rate found earlier
best_optimizer_fn = lambda model: torch.optim.Adam(model.parameters(), lr=best_learning_rate)  # Replace with the best optimizer
best_activation_fn = nn.ReLU()  # Replace with the best activation function found earlier

results = {"Channel_1": [], "Train Accuracy": [], "Validation Accuracy": []}

# model and record the train and validation accuracy for each configuration.   #
for channel_1, channel_2 in channel_sizes:

    model = nn.Sequential(
        nn.Conv2d(in_channels=in_channel, out_channels=channel_1, kernel_size=5, padding=2),
        best_activation_fn,
        nn.Conv2d(in_channels=channel_1, out_channels=channel_2, kernel_size=3, padding=1),
        best_activation_fn,
        nn.Flatten(start_dim=1, end_dim=-1),
        nn.Linear(in_features=channel_2 * 32 * 32, out_features=num_classes)
    )
    model.to(device=device)
    optimizer = best_optimizer_fn(model)

    print(f"\n------Testing with: CH1: {channel_1}, CH2: {channel_2}.------")
    epoch_loss = train_part3(model, optimizer)

    train_acc = check_accuracy_part3(train_loader, model)
    val_acc = check_accuracy_part3(val_loader, model)

    results['Channel_1'].append(channel_1)
    results['Train Accuracy'].append(train_acc)
    results['Validation Accuracy'].append(val_acc)


In [ ]:
import matplotlib.pyplot as plt

def plot_channel_size_results(results):
    plt.figure(figsize=(10, 6))

    # Scatter plot for Train Accuracy
    plt.scatter(results["Channel_1"], results["Train Accuracy"], label="Train Accuracy", color='blue')

    # Scatter plot for Validation Accuracy
    plt.scatter(results["Channel_1"], results["Validation Accuracy"], label="Validation Accuracy", color='orange')

    # Plot dotted lines between train and validation accuracy for each channel size
    for i in range(len(results["Channel_1"])):
        plt.plot([results["Channel_1"][i], results["Channel_1"][i]],
                 [results["Train Accuracy"][i], results["Validation Accuracy"][i]],
                 linestyle=':', color='gray')  # Dotted line between points

    # Optional: Use a logarithmic scale for the x-axis if necessary
    # plt.xscale("log")  # Log scale for better visibility of channel sizes

    # Add labels and title
    plt.xlabel("Channel_1 (First Layer)")
    plt.ylabel("Accuracy (%)")
    plt.title("Effect of Channel Sizes on Train and Validation Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

# Example usage
plot_channel_size_results(results)


### Sequential API: Evaluate the final ConvNet model on the test set of CIFAR-10


**Implementation note.**

Using the best configuration found during the previous experiments (optimizer, learning rate, activation function, and channel sizes), train the model for 5 epoch and evaluate its performance on the test set.

1.	Report the test accuracy.

2.	Compare it with the validation accuracy to assess if your model generalized well.


In [ ]:
# the previous experiments and evaluate its performance on the test set. Train #
# the model for 5 epochs and report the test accuracy.                         #
#                                                                              #
# NOTE: Use the best optimizer, learning rate, activation function, and        #
# channel sizes found earlier.                                                 #
best_learning_rate = 0.001  # Replace with the best learning rate found earlier
best_optimizer_fn = lambda model: torch.optim.Adam(model.parameters(), lr=best_learning_rate)  # Replace with the best optimizer
best_activation_fn = nn.ReLU()  # Replace with the best activation function found earlier
best_channel_size = (32, 16)

epochs = 5
model = nn.Sequential(
    nn.Conv2d(in_channels=in_channel, out_channels=channel_1, kernel_size=5, padding=2),
    best_activation_fn,
    nn.Conv2d(in_channels=channel_1, out_channels=channel_2, kernel_size=3, padding=1),
    best_activation_fn,
    nn.Flatten(start_dim=1, end_dim=-1),
    nn.Linear(in_features=channel_2 * 32 * 32, out_features=num_classes)
)
optimizer = best_optimizer_fn(model)

for e in range(epochs):
    _ = train_part3(model, optimizer)

check_accuracy_part3(test_loader, model)


## Train ResNet-18 from scratch


**Implementation note.**

So far, we’ve tried to design and train a **custom** three-layer convolutional network to the best of our ability.

However, in the literature, there are **well-known** convolutional architectures that are highly effective for image classification tasks.

This section:

1.	Load a pre-defined [ResNet-18](https://pytorch.org/vision/main/models/generated/torchvision.models.resnet18.html) model from the `torchvision.models` library.

2.	Modify it for CIFAR-10 classification (10 output classes).

3.	Train it from scratch for 5 epochs using the train_part3 function.

4.	Report the validation accuracy after each epoch and evaluate the test accuracy at the end.

NOTE: Make sure to initialize the optimizer with a `learning rate` of `1e-3` and use the `AdamW` optimizer.


In [ ]:
import torchvision.models as models

# train it from scratch for 5 epochs, and evaluate on the test set.            #
model = models.resnet18(weights=None)  # Initialize the model without pre-trained weights
model.fc = nn.Linear(model.fc.in_features, 10)
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

results = {}
for e in range(epochs):
    _ = train_part3(model, optimizer)

    val_acc = check_accuracy_part3(val_loader, model)

    results[e] = val_acc


for epoch, val_acc in results.items():
    print(f"Epoch {epoch}, Val Acc: {val_acc}")

check_accuracy_part3(test_loader, model)


## Fine-tune a pre-trained ResNet-18


#### Fine-tune only the classifier


**Implementation note.**

This section explore **transfer learning** by leveraging a pretrained ResNet-18 model. Instead of training the entire model from scratch, you will:

1.	Load a [ResNet-18](https://pytorch.org/vision/main/models/generated/torchvision.models.resnet18.html) model pretrained on [ImageNet-1k](https://www.image-net.org/download.php) using `torchvision.models`.

2.	Replace the final fully connected layer to output 10 classes (for CIFAR-10).

3.	Freeze all layers except the classifier.

4.	Train the model for 5 epochs, fine-tuning only the classifier, and evaluate its performance on the test set.

NOTE: Use the `AdamW` optimizer with a `learning rate` of `1e-2`.


In [ ]:
import torchvision.models as models

# the final layer for CIFAR-10 classification, freeze all other layers, and   #
# fine-tune the classifier for 5 epochs. Evaluate the model on the test set.  #
# Step 1: Load the pre-trained ResNet-18 model

model = models.resnet18(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 10)
params_to_update = model.parameters()
optimizer = optim.AdamW(model.parameters(), lr=1e-2)

fine_tuning_last_fc = True

if fine_tuning_last_fc:
        # Step 3: Freeze all layers except the final one
    for param in model.parameters():
        param.requires_grad = False

    # Allow the final layer to be trainable
    for param in model.fc.parameters():
        param.requires_grad = True

print("Params to learn:")
if fine_tuning_last_fc:
    params_to_update = []
    for name,param in model.named_parameters():
        if param.requires_grad == True:
            params_to_update.append(param)
            print("\t",name)

else:
    for name,param in model.named_parameters():
        if param.requires_grad == True:
            print("\t",name)


results = {}
for e in range(epochs):
    _ = train_part3(model, optimizer)

    val_acc = check_accuracy_part3(val_loader, model)

    results[e] = val_acc


for epoch, val_acc in results.items():
    print(f"Epoch {epoch}, Val Acc: {val_acc}")

check_accuracy_part3(test_loader, model)


#### Fine-tune the whole network


**Implementation note.**

This section fine-tune the entire pretrained ResNet-18 model for CIFAR-10 classification. Instead of freezing the backbone, you will:

1.	Load a [ResNet-18](https://pytorch.org/vision/main/models/generated/torchvision.models.resnet18.html) model pretrained on ImageNet using `torchvision.models`.

2.	Replace the final fully connected layer to output 10 classes (for CIFAR-10).

3.	Fine-tune all layers of the model (i.e., allow updates to the entire network).

4.	Train the model for 5 epochs and evaluate its performance on the test set.

NOTE: Use the `AdamW` optimizer with a `learning rate` of `1e-3`.


In [ ]:
import torchvision.models as models

# the final layer for CIFAR-10 classification and update all model layers.    #
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 10)


# Step 5: Set up the AdamW optimizer and loss function
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

results = {}
for e in range(epochs):
    _ = train_part3(model, optimizer)

    val_acc = check_accuracy_part3(val_loader, model)

    results[e] = val_acc


for epoch, val_acc in results.items():
    print(f"Epoch {epoch}, Val Acc: {val_acc}")

check_accuracy_part3(test_loader, model)


## Technical Report


**Concept check.**

Summarize your findings from the experiments with the custom ConvNet and ResNet-18 (trained from scratch and fine-tuned). Your report should include:

1.	Performance Comparison: Compare test accuracies of all approaches. Which performed best and why?

2.	Efficiency: Discuss training time and resource usage for each approach. Which was the most efficient?

3.	Recommendations: Recommend the best approach for CIFAR-10 and justify your choice based on accuracy, efficiency, and ease of implementation.


**Answer.**
1. The 2-layer custom ConvNet showed incremental improvements with hyperparameter tuning but never outperformed the ResNet-18 (not pretrained) and the pretrained ResNet-18 with fine-tuning an all layers. The pretrained ResNet-18 (with only the final FC layer replaced) performed poorly on CIFAR-10, likely due to the mismatch between the 1000 ImageNet classes and CIFAR-10's 10 classes. In contrast, the fine-tuned ResNet-18, where all layers were unfrozen and retrained on CIFAR-10, delivered the best performance.
2. There were no significant differences in training time between the models.
3. The fine-tuned (all layers) ResNet-18 is the best approach for CIFAR-10 due to its ability to leverage pretrained feature extraction from ImageNet, which significantly improves performance. Pretrained models like ResNet-18 have already learned rich image representations, making them highly effective for transfer learning on smaller datasets like CIFAR-10.
